# Nmap LLM Fine-tuning

Fine-tune Qwen to execute nmap scan workflows.

**Setup:**
1. Runtime → Change runtime → GPU
2. Edit cell 4 with your GitHub repo
3. Run all cells in order


In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT: Replace with your GitHub repo
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/nmap-llama

In [ ]:
exec(open('/content/nmap-llama/finetune/generate_training_data.py').read())
main()

In [ ]:
import os, json, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
MODEL_NAME = 'Qwen/Qwen2-0.5B'
OUTPUT_DIR = '/content/drive/MyDrive/nmap-finetuned'
BATCH_SIZE = 2
LEARNING_RATE = 2e-4
NUM_EPOCHS = 5
MAX_LENGTH = 768

In [ ]:
DATA_PATH = '/content/nmap-llama/finetune/data/training_qwen.txt'
texts = []
with open(DATA_PATH) as f:
    for line in f:
        texts.append(line.strip())
dataset = Dataset.from_dict({'text': texts})
print(f'Loaded {len(dataset)} examples')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer ready!')

In [ ]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
print('Model loaded!')

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize(examples):
    result = tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result
tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
print(f'Tokenized: {len(tokenized)}')

In [ ]:
training_args = TrainingArguments(output_dir=OUTPUT_DIR, per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=4, learning_rate=LEARNING_RATE, num_train_epochs=NUM_EPOCHS, logging_steps=10, save_strategy='epoch', fp16=True, report_to='none', warmup_steps=20, optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
print('Ready to train!')

In [ ]:
print('Starting training...')
trainer.train()
print('Done!')

In [ ]:
trainer.save_model(f'{OUTPUT_DIR}/final')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/final')
print(f'Saved to {OUTPUT_DIR}/final')

In [ ]:
from transformers import pipeline
pipe = pipeline('text-generation', model=f'{OUTPUT_DIR}/final', tokenizer=f'{OUTPUT_DIR}/final')
prompts = ['Target: 192.168.1.0/24\nType: CIDR\nGoal: Full scan']
for p in prompts:
    print(f'\nPROMPT: {p}')
    print(pipe(p, max_new_tokens=200)[0]['generated_text'])